# Notebook 5: MLflow Model Registry

**Learning Objectives:**
- Register models in MLflow Model Registry
- Manage model versions
- Use model aliases (champion, challenger) to manage deployment stages
- Load and use registered models
- Update model metadata and descriptions

**Dataset:** Wine classification

> **Note:** MLflow 3.x replaced the old stages API (`Staging`, `Production`, `Archived`)
> with **model aliases**. This notebook uses the modern aliases approach.

In [12]:
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# ---------- workaround for MLflow 3.x FileStore YAML bug ----------
# MLflow 3.x stores Metric/Param objects in model-version metadata,
# but PyYAML cannot serialize them.  Adding a safe representer fixes it.
import yaml
from mlflow.entities import Metric, Param

def _represent_metric(dumper, data):
    return dumper.represent_dict({
        "key": data.key, "value": data.value,
        "timestamp": data.timestamp, "step": data.step,
    })

def _represent_param(dumper, data):
    return dumper.represent_dict({"key": data.key, "value": data.value})

yaml.representer.SafeRepresenter.add_representer(Metric, _represent_metric)
yaml.representer.SafeRepresenter.add_representer(Param, _represent_param)
# -------------------------------------------------------------------

## Step 1: Prepare Data

In [13]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset: {X.shape}")
print(f"Classes: {wine.target_names}")

Dataset: (178, 13)
Classes: ['class_0' 'class_1' 'class_2']


## Step 2: Train and Register First Model

In [14]:
mlflow.set_experiment("05_Model_Registry")

with mlflow.start_run(run_name="Wine_Classifier_v1") as run:
    
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Log parameters and metrics
    mlflow.log_params({"n_estimators": 100, "max_depth": 5})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Register model in Model Registry
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    run_id = run.info.run_id
    
    print(f" Model registered: WineClassifier")
    print(f"   Version: 1")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")
    print(f"   Run ID: {run_id}")

2026/02/15 00:53:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/15 00:53:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'WineClassifier' already exists. Creating a new version of this model...
Created version '4' of model 'WineClassifier'.


 Model registered: WineClassifier
   Version: 1
   Accuracy: 1.0000
   F1 Score: 1.0000
   Run ID: 2b4c1df6d0074633a9dc2454ebd70658


## Step 3: Train and Register Second Model Version

In [15]:
with mlflow.start_run(run_name="Wine_Classifier_v2"):
    
    model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    mlflow.log_params({"n_estimators": 200, "max_depth": 10})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    print(f" New model version registered: WineClassifier")
    print(f"   Version: 2")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")

2026/02/15 00:53:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/15 00:53:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'WineClassifier' already exists. Creating a new version of this model...


 New model version registered: WineClassifier
   Version: 2
   Accuracy: 1.0000
   F1 Score: 1.0000


Created version '5' of model 'WineClassifier'.


## Step 4: Train Different Algorithm (Version 3)

In [16]:
with mlflow.start_run(run_name="Wine_Classifier_v3_GB"):
    
    model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    mlflow.log_params({"algorithm": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.1})
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score", f1)
    
    # Register as version 3
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="WineClassifier"
    )
    
    print(f" Third model version registered: WineClassifier")
    print(f"   Version: 3 (Gradient Boosting)")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   F1 Score: {f1:.4f}")

2026/02/15 00:53:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/15 00:53:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'WineClassifier' already exists. Creating a new version of this model...


 Third model version registered: WineClassifier
   Version: 3 (Gradient Boosting)
   Accuracy: 0.9444
   F1 Score: 0.9443


Created version '6' of model 'WineClassifier'.


## Step 5: Manage Model Versions with MlflowClient

In [17]:
client = MlflowClient()

model_name = "WineClassifier"
versions = client.search_model_versions(f"name='{model_name}'")

print("\n" + "="*70)
print(f"ALL VERSIONS OF '{model_name}'")
print("="*70)

for v in versions:
    print(f"\nVersion: {v.version}")
    print(f"  Aliases: {v.aliases}")
    print(f"  Run ID: {v.run_id}")
    print(f"  Created: {v.creation_timestamp}")


ALL VERSIONS OF 'WineClassifier'

Version: 6
  Aliases: []
  Run ID: f38a07edbb354bbaac57d23e52376411
  Created: 1771109595777

Version: 5
  Aliases: []
  Run ID: eaa89d14aeb34a22b4d62223a21e1070
  Created: 1771109592369

Version: 4
  Aliases: []
  Run ID: 2b4c1df6d0074633a9dc2454ebd70658
  Created: 1771109589089

Version: 3
  Aliases: ['challenger']
  Run ID: aa99d8a02b9a4d95bdc3c05bcaa82fb1
  Created: 1771109510250

Version: 2
  Aliases: ['champion']
  Run ID: 38b6999689744d5893cf05eca3c497d3
  Created: 1771109502628

Version: 1
  Aliases: []
  Run ID: f04104d63a3d4d78890253a1c243eedc
  Created: 1771109487765


## Step 6: Assign Model Aliases

> In MLflow 3.x, model **aliases** (e.g. `champion`, `challenger`) replace the old
> stages (`Staging`, `Production`, `Archived`).

In [18]:
# Assign aliases to model versions
client.set_registered_model_alias(
    name="WineClassifier",
    alias="challenger",
    version=1
)
print("Version 1 → challenger")

client.set_registered_model_alias(
    name="WineClassifier",
    alias="champion",
    version=2
)
print("Version 2 → champion")

# Version 3 has no alias yet (under evaluation)
print("Version 3 → no alias (under evaluation)")

print("\nModel lifecycle with aliases:")
print("   (no alias) → challenger → champion")

Version 1 → challenger
Version 2 → champion
Version 3 → no alias (under evaluation)

Model lifecycle with aliases:
   (no alias) → challenger → champion


## Step 7: Update Model Descriptions

In [19]:
client.update_model_version(
    name="WineClassifier",
    version=1,
    description="Baseline RandomForest model with 100 trees, max_depth=5. Good starting point."
)

client.update_model_version(
    name="WineClassifier",
    version=2,
    description="Improved RandomForest with 200 trees, max_depth=10. Current production model."
)

client.update_model_version(
    name="WineClassifier",
    version=3,
    description="Experimental GradientBoosting model. Under evaluation for production."
)

print(" Descriptions updated for all versions")

 Descriptions updated for all versions


## Step 8: Load and Use Registered Models

In [20]:
# Load specific version
model_v1 = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier/1")
print("Loaded Version 1")

# Load model by alias
champion_model = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier@champion")
print("Loaded champion model")

challenger_model = mlflow.pyfunc.load_model(model_uri="models:/WineClassifier@challenger")
print("Loaded challenger model")

predictions = champion_model.predict(X_test[:5])
print(f"\nChampion model predictions (first 5 samples):")
print(predictions)
print(f"\nTrue labels:")
print(y_test[:5])

Loaded Version 1
Loaded champion model
Loaded challenger model

Champion model predictions (first 5 samples):
[0 2 0 1 1]

True labels:
[0 2 0 1 1]


## Step 9: Compare Model Versions

In [21]:
# Get metrics for all versions
results = []

for ver in [1, 2, 3]:
    version_info = client.get_model_version("WineClassifier", ver)
    run = client.get_run(version_info.run_id)

    results.append({
        "version": ver,
        "aliases": ", ".join(version_info.aliases) if version_info.aliases else "none",
        "accuracy": float(run.data.metrics.get("accuracy", 0)),
        "f1_score": float(run.data.metrics.get("f1_score", 0))
    })

df_comparison = pd.DataFrame(results)
print("\n" + "="*70)
print("MODEL VERSION COMPARISON")
print("="*70)
print(df_comparison.to_string(index=False))

print("\nVersion 2 is the champion with best performance!")


MODEL VERSION COMPARISON
 version    aliases  accuracy  f1_score
       1 challenger  1.000000  1.000000
       2   champion  1.000000  1.000000
       3       none  0.944444  0.944269

Version 2 is the champion with best performance!


## Step 10: Reassign Aliases

In [22]:
# Remove the challenger alias from version 1
client.delete_registered_model_alias(
    name="WineClassifier",
    alias="challenger"
)
print("Removed 'challenger' alias from version 1")

# Assign challenger alias to version 3 for evaluation
client.set_registered_model_alias(
    name="WineClassifier",
    alias="challenger",
    version=3
)
print("Version 3 → challenger")

print("\nCurrent model aliases:")
print("   Version 1: (no alias - old baseline)")
print("   Version 2: champion (current best)")
print("   Version 3: challenger (under evaluation)")

Removed 'challenger' alias from version 1
Version 3 → challenger

Current model aliases:
   Version 1: (no alias - old baseline)
   Version 2: champion (current best)
   Version 3: challenger (under evaluation)


## Key Takeaways

1. **Model Registry** - Centralized model management
2. **Versioning** - Automatic version tracking
3. **Aliases** - `champion`, `challenger`, or any custom alias to tag versions
4. **Loading Models** - By version number or by alias
5. **Metadata** - Descriptions, tags for documentation
6. **Comparison** - Easy comparison between versions

## Exercise

1. Go to MLflow UI → Models tab
2. Click on "WineClassifier"
3. View all 3 versions and their aliases
4. Try reassigning `challenger` to a different version
5. Compare metrics across versions

## Production Workflow

```python
# In production code, always use the champion alias:
model = mlflow.pyfunc.load_model("models:/WineClassifier@champion")
predictions = model.predict(new_data)
```

This ensures you're always using the approved production model!